# Bowler-swap check (Nov 3, 2026)

**About this file.** I ran this check in Google Colab on Nov 3, 2026, in a notebook I later reused for the batter-swap check, which overwrote these cells. This file puts back the exact code (BS0-BS4) from my project notes. **The cells here have not been re-run, so they have no saved outputs.** The results under each cell are the numbers I recorded on Nov 3 from that run. Re-running it needs `ipl_features.parquet`, `feature_list.json` and `xgb_X-hist_2026.json` in `DATA` (from `features.ipynb` and `test_2026.ipynb`); it takes about 15-20 minutes on CPU.

This check is why I dropped the bowling simulator (README, *Decisions*; website, *Limitations* and *AI Assistance*).

## The question

A game where the user picks who bowls overs 17-20 only works if *who bowls* moves XGBoost's predictions. Bowler history was the weakest feature group, so the worry was that every bowler looks the same to the model.

## The method

Take every 2026 death ball in a chase (innings 2, overs 17-20). Keep the match state exactly as it was, swap in a different bowler, re-predict, and repeat for every regular 2026 death bowler. Because every bowler faces the same balls, the differences come only from identity.

Identity means 14 columns: the 8 bowler history features (`bowl_*`), `bowler_cat`, `matchup_type`, and the 4 base rates that follow from them. The in-match bowler figures stay as they were on the real ball. Each bowler's values are the average of his own 2026 death rows, split by batter hand.

## Rules, set before seeing the numbers

1. **League spread (BS2):** the 10th-to-90th percentile gap in predicted runs per over among regular death bowlers. Below 0.5 = flat; 1.5 or more = worth simulating.
2. **Inside one match (BS3, BS4):** the gap between the cheapest and dearest plan for overs 17-20. Median of 2 runs or more = build the game; below 1 run = flat.

Why 2 runs: near a 50-50 finish, runs in the last 4 overs have an SD of about 10, so one expected run is worth about 4 points of win probability.

## BS0. Setup and load

In [ ]:
# ===== BS0. Setup and load (COPY-PASTE) =====
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'

df = pd.read_parquet(DATA + 'ipl_features.parquet')
with open(DATA + 'feature_list.json') as f: meta = json.load(f)
FEATURES, CAT_FEATURES = meta['features'], meta['categorical']
HIST = sum(meta['hist_features'].values(), [])
FEAT_HIST = list(FEATURES) + HIST                     # the 87 X-hist inputs, frozen order
RUNS = np.array([0, 1, 2, 3, 4, 6, 0])                # runs for each class (0,1,2,3,4,6,W)

# same rows and category codes as test_2026 T0 (the saved model needs them)
d = df[(df['y'] >= 0) & df['split'].isin(['train', 'val', 'test'])].copy()
d['y'] = d['y'].astype(int)
for c in CAT_FEATURES:
    if not isinstance(d[c].dtype, pd.CategoricalDtype):
        d[c] = d[c].astype('category')

bst = xgb.Booster(); bst.load_model(DATA + 'xgb_X-hist_2026.json')
bst.set_param({'device': 'cpu'})
def predict(X):
    return bst.predict(xgb.DMatrix(X[FEAT_HIST], enable_categorical=True))   # (n, 7) probabilities

death26 = d[(d['split'] == 'test') & (d['match_phase'] == 'death')]
chase = death26[death26['innings'] == 2]
print('1. 2026 death balls:', len(death26), '| in chases:', len(chase),
      '| chase matches:', chase['match_id'].nunique())                  # expect 3583 | about 1600-1800 | about 70
P0 = predict(chase)
print('2. real bowlers, runs per over: model', round(6 * (P0 @ RUNS).mean(), 2),
      '| actual', round(6 * RUNS[chase['y']].mean(), 2))               # expect the two within about 0.5

**Recorded result (Nov 3):** 3,583 death balls in 2026, 1,494 in chases, 66 chase matches. Runs per over with the real bowlers: model 10.46, actual 10.17.

## BS1. Each bowler's identity values, and the swap

In [ ]:
# ===== BS1. Identity values and the swap function (TYPE IT) =====
SWAP_NUM = meta['hist_features']['9_bowler'] + ['base_econ_cat_phase', 'base_wkt_pct_cat_phase',
                                                'base_bdry_pct_mp', 'base_wkt_pct_mp']
SWAP_CAT = ['bowler_cat', 'matchup_type']

# regular 2026 death bowlers: 24+ death balls (4 overs) and faced both batter hands
n_death = death26['bowler'].value_counts()
n_hands = death26.groupby('bowler', observed=True)['batter_hand'].nunique().reindex(n_death.index)
pool = n_death[(n_death >= 24) & (n_hands >= 2)].index.tolist()
print('3. bowlers with 24+ death balls:', int((n_death >= 24).sum()), '| kept (both hands):', len(pool))

# one row of identity values per (bowler, batter hand): the mean over his 2026 death balls
rows = death26[death26['bowler'].isin(pool)]
donor_num = rows.groupby(['bowler', 'batter_hand'], observed=True)[SWAP_NUM].mean()
donor_cat = rows.groupby(['bowler', 'batter_hand'], observed=True)[SWAP_CAT].agg(lambda s: s.mode().iloc[0])

def swap_bowler(balls, b):
    """The same balls and match state, with bowler b's identity features."""
    X = balls.copy()
    hand = X['batter_hand'].astype(str).to_numpy()
    num = donor_num.loc[b].rename(index=str).reindex(hand)       # one row per ball, matched on batter hand
    cat = donor_cat.loc[b].rename(index=str).reindex(hand)
    assert not num.isna().any().any(), f'{b}: no values for some batter hand'
    X[SWAP_NUM] = num.to_numpy()
    for c in SWAP_CAT:
        X[c] = pd.Categorical(cat[c].to_numpy(), categories=balls[c].cat.categories)
    return X

# self-swap test: put a bowler onto his own balls; only the match-to-match drift of history should change
b = pool[0]
own = death26[death26['bowler'] == b]
gap = np.abs(predict(swap_bowler(own, b)) - predict(own)).mean()
print('4. self-swap', b, '| mean |change in p|:', round(gap, 4))      # expect below 0.005

**Recorded result (Nov 3):** 55 bowlers with 24+ death balls, 54 kept (faced both batter hands). Self-swap test (B Kumar onto his own balls): mean |change in p| = 0.0016, below the 0.005 check.

## BS2. Every bowler on the same chase balls (rule 1)

In [ ]:
# ===== BS2. League spread (COPY-PASTE) =====
ER, EW, E6 = {}, {}, {}
for b in pool:
    P = predict(swap_bowler(chase, b))
    ER[b], E6[b], EW[b] = P @ RUNS, P[:, 5], P[:, 6]
ER, E6, EW = (pd.DataFrame(D, index=chase.index) for D in (ER, E6, EW))

tab = pd.DataFrame({'rpo': 6 * ER.mean(), 'P6 %': 100 * E6.mean(), 'PW %': 100 * EW.mean(),
                    'hist death econ': donor_num.groupby(level=0)['bowl_econ_phase'].mean(),
                    'death balls 2026': n_death}, index=pool).sort_values('rpo')
print(tab.round(2).to_string())

q = tab['rpo'].quantile([0.1, 0.5, 0.9])
print(f"\n5. predicted rpo: best {tab['rpo'].min():.2f} | p10 {q[0.1]:.2f} | median {q[0.5]:.2f} "
      f"| p90 {q[0.9]:.2f} | worst {tab['rpo'].max():.2f}")
print('6. p90 - p10:', round(q[0.9] - q[0.1], 2), 'runs per over   (rule: < 0.5 flat, >= 1.5 worth it)')
print('7. wickets per over p10-p90:', (6 * EW.mean()).quantile([0.1, 0.9]).round(3).tolist())
print('8. Spearman rpo vs history death economy:',
      round(tab['rpo'].corr(tab['hist death econ'], method='spearman'), 2))   # expect clearly positive

**Recorded result (Nov 3):** predicted runs per over: best 9.05, p10 10.21, median 10.43, p90 10.74, worst 11.21. **p90 − p10 = 0.52**, on the flat line. Wickets per over p10-p90: 0.469-0.523. Spearman with history death economy: 0.19.

## BS3. What the choice is worth inside one match (rule 2, ignoring overs already bowled)

In [ ]:
# ===== BS3. Inside one match (COPY-PASTE) =====
in_pool = set(pool)
out = []
for m, g in chase.groupby('match_id', observed=True):
    bowled = d.loc[(d['match_id'] == m) & (d['innings'] == 2), 'bowler'].unique()
    opts = [b for b in bowled if b in in_pool]
    if len(opts) < 3 or len(g) < 12:
        continue
    r = (6 * ER.loc[g.index, opts].mean()).sort_values()       # runs per over for each option, cheapest first
    w = 6 * EW.loc[g.index, opts].mean()                        # wickets per over
    out.append({'match_id': m, 'options': len(opts), 'death balls': len(g),
                'spread rpo': r.iloc[-1] - r.iloc[0],
                'plan gap runs': 2 * (r.iloc[-2:].sum() - r.iloc[:2].sum()),
                'plan gap wkts': 2 * (w[r.index[:2]].sum() - w[r.index[-2:]].sum()),
                'cheapest': ', '.join(r.index[:2]), 'dearest': ', '.join(r.index[-2:])})
mt = pd.DataFrame(out).sort_values('plan gap runs', ascending=False)

print('9. matches with 3+ options:', len(mt), 'of', chase['match_id'].nunique(),
      '| options per match (median):', mt['options'].median())
print('10. plan gap in runs: p25 / median / p75 =', mt['plan gap runs'].quantile([.25, .5, .75]).round(2).tolist(),
      '  (rule: median >= 2 build, < 1 flat)')
print('11. share of matches with a gap of 2+ runs:', round((mt['plan gap runs'] >= 2).mean(), 2))
print('12. plan gap in wickets (cheap plan minus dear plan), median:', round(mt['plan gap wkts'].median(), 3))
print(mt.head(8).round(2).to_string(index=False))

**Recorded result (Nov 3):** 50 of 66 chases have 3+ options (median 4). Plan gap in runs p25 / median / p75 = 0.71 / 1.45 / 2.13, the grey zone. 28% of matches have a gap of 2+ runs. Median wicket gap 0.028.

## BS4. With real quotas at the start of over 17

In [ ]:
# ===== BS4. Are the big-gap matches playable at over 17? (COPY-PASTE) =====
def plan_runs(r, left, cheapest=True):
    """Expected runs in overs 17-20: fill the 4 overs from the cheapest (or dearest) bowler first,
    at most 2 overs each (no back-to-back overs) and no more than his overs left."""
    total, n = 0.0, 0
    for b in (r.index if cheapest else r.index[::-1]):
        k = int(min(2, left[b], 4 - n))
        total, n = total + k * r[b], n + k
        if n == 4:
            return total
    return np.nan                                              # these bowlers don't have 4 overs between them

inn2 = d[(d['split'] == 'test') & (d['innings'] == 2)]
res = []
for m, g in inn2.groupby('match_id', observed=True):
    if g['c_team_balls'].max() < 96:
        continue                                               # chase finished before over 17
    s = g[g['c_team_balls'] >= 96].sort_values('c_team_balls', kind='stable').iloc[0]   # first ball of over 17
    before = g[g['c_team_balls'] < 96]
    used = np.ceil(before.groupby('bowler', observed=True).size() / 6)
    everyone = pd.Index(g['bowler'].unique())                  # every bowler used in this innings
    left = (4 - used.reindex(everyone, fill_value=0)).clip(lower=0)
    can = left[left >= 1]                                      # bowlers who could still bowl at over 17
    opts = [b for b in can.index if b in in_pool]
    dg = g[g['c_team_balls'] >= 96].index.intersection(chase.index)
    r = (6 * ER.loc[dg, opts].mean()).sort_values() if opts else pd.Series(dtype=float)
    best, worst = plan_runs(r, left, True), plan_runs(r, left, False)
    res.append({'match_id': m, 'need': int(s['runs_needed']), 'wkts down': int(s['c_team_wickets']),
                'can bowl': len(can), 'in pool': len(opts), 'overs left (pool)': int(left[opts].sum()),
                'real gap runs': worst - best,
                'options: overs left, rpo': ', '.join(f'{b} {int(left[b])}ov {r[b]:.1f}' for b in r.index)})
pl = pd.DataFrame(res).sort_values('real gap runs', ascending=False)
pl['close?'] = pl['need'].between(28, 52) & (pl['wkts down'] <= 6)    # rough proxy for a 25-75% chase

print('13. chases that reach over 17:', len(pl), '| with 4+ pool overs left:', int(pl['real gap runs'].notna().sum()))
print('14. real gap in runs (quotas respected): p25 / median / p75 =',
      pl['real gap runs'].quantile([.25, .5, .75]).round(2).tolist())
print('15. close by the rough proxy:', int(pl['close?'].sum()),
      '| close AND real gap >= 2 runs:', int((pl['close?'] & (pl['real gap runs'] >= 2)).sum()))
print(pl.head(12).round(2).to_string(index=False))

**Recorded result (Nov 3):** 61 chases reach over 17; 38 have 4+ overs left among the regular death bowlers. Gap between the best and worst plans for overs 17-20 with quotas respected: p25 0.00 / **median 0.11** / p75 0.60 runs, max 1.80. Close chases by the rough proxy: 25; close and a gap of 2+ runs: 0.

## Verdict

The bowler-choice game fails the rule set in advance once real quotas are respected: the median gap is 0.11 runs, below the 1-run flat line, and no close chase has a 2-run choice. BS3's bigger gaps came from bowlers who had in reality already used their overs before over 17. To XGBoost, the death-bowling choice still open at over 17 is worth about 0.1 runs, or about half a point of win probability. So I dropped the simulator.

**Simplifications** (fine for a go/no-go check): identity values are a bowler's 2026 average, not the value for that match; in-match bowler figures are held at the real bowler's values; only regular death bowlers (24+ death balls) are options; runs are off-the-bat only, like the model's target.